# Ergasia AI: Local QLoRA Fine-Tuning (`Qwen2.5-7B-Instruct`)

This notebook fine-tunes **Qwen2.5-7B-Instruct** in 4-bit (QLoRA) using **Unsloth** on a Windows 11 Workstation (RTX 4090, 24 GB VRAM) to produce two hot-swappable LoRA adapters:
1. `lora_generation_adapter`: Generates CO-PO mapped, rubric-backed questions calibrated to student BKT mastery levels.
2. `lora_grading_adapter`: Evaluates student submissions against the generated rubric and produces structured scores and feedback.

## Step 0: Optional `torchao` Downgrade
If you want to permanently downgrade `torchao` in your environment without breaking CUDA 12.4 PyTorch, uncomment and run the cell below. However, **Step 1 also includes a runtime monkey-patch** so the notebook will work immediately even without downgrading.

In [6]:
# !pip install "torchao<0.16.0" --no-deps

## Step 1: Environment Setup & Windows Workarounds
- Disables Triton inductor compilation on Windows (`UNSLOTH_COMPILE_DISABLE=1`).
- Locks execution to the discrete RTX 4090 GPU (`CUDA_VISIBLE_DEVICES=0`).
- Shims `torch.utils._pytree.register_constant` so `torchao` (v0.17+) does not crash on stable PyTorch builds.

In [7]:
import os
import gc
import json

# 1. Disable Triton compilation to prevent Windows crashes
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
# 2. Target discrete GPU explicitly
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

# 3. Monkey-patch _pytree for torchao v0.17+ compatibility on stable PyTorch (CUDA 12.4)
if not hasattr(torch.utils._pytree, "register_constant"):
    torch.utils._pytree.register_constant = lambda *args, **kwargs: None

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

PyTorch Version: 2.6.0+cu124
CUDA Available: True
Device Name: NVIDIA GeForce RTX 4090
BF16 Supported: True


## Step 2: Inspect Local `.jsonl` Datasets
Verifies that `generation_data.jsonl` and `grading_data.jsonl` exist in the workspace root and auto-detects the conversation key (`messages` vs `conversations`).

In [8]:
for filename in ["generation_data.jsonl", "grading_data.jsonl"]:
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as f:
            first_line = json.loads(f.readline())
            total_lines = 1 + sum(1 for _ in f)
        print(f"[OK] {filename}: {total_lines} examples | Top-level keys: {list(first_line.keys())}")
    else:
        print(f"[WARNING] {filename} not found in {os.getcwd()}")

[OK] generation_data.jsonl: 18 examples | Top-level keys: ['text']
[OK] grading_data.jsonl: 18 examples | Top-level keys: ['text']


## Step 3: Define Reusable Model Loader & Training Pipeline
To prevent adapter weight contamination (training the grading adapter on top of the already modified generation adapter weights), this helper function loads a fresh 4-bit base model and attaches a clean LoRA configuration for each run.

In [9]:
MAX_SEQ_LENGTH = 2048
BASE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

def load_fresh_model_and_tokenizer():
    """Loads a fresh 4-bit Qwen2.5-7B-Instruct base model and attaches clean LoRA weights."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = BASE_MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype = None,       # Auto-detects bfloat16 on RTX 4090
        load_in_4bit = True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha = 16,
        lora_dropout = 0,   # Optimized for Unsloth fast patching
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "chatml",
        mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
    )
    return model, tokenizer


def train_ergasia_adapter(jsonl_path, output_adapter_dir, num_train_epochs=3, max_steps=-1):
    """Loads dataset, auto-detects JSONL schema, trains a LoRA adapter, saves weights, and frees VRAM."""
    print(f"\n{'='*60}\nStarting training for: {output_adapter_dir}\n{'='*60}")
    
    # Clear any leftover model in VRAM from an interrupted run
    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = load_fresh_model_and_tokenizer()
    dataset = load_dataset("json", data_files=jsonl_path, split="train")
    cols = dataset.column_names
    print(f"Detected columns in {jsonl_path}: {cols}")

    def format_prompts(examples):
        # Case 1: Already has a pre-formatted 'text' column
        if "text" in cols and isinstance(examples["text"][0], str):
            return {"text": examples["text"]}

        # Case 2: ChatML list of dicts under any column name (e.g., messages, conversations, chat, history)
        for col in cols:
            if isinstance(examples[col][0], list):
                convos = examples[col]
                # Handle ShareGPT ('from'/'value') vs OpenAI ('role'/'content') keys inside the list
                first_turn = convos[0][0] if len(convos[0]) > 0 else {}
                if "from" in first_turn and "value" in first_turn:
                    normalized = [
                        [{"role": ("user" if t["from"] in ["human", "user"] else ("assistant" if t["from"] in ["gpt", "assistant", "model"] else "system")), "content": t["value"]} for t in c]
                        for c in convos
                    ]
                else:
                    normalized = convos
                texts = [
                    tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
                    for c in normalized
                ]
                return {"text": texts}

        # Case 3: Instruction / Input / Output (Alpaca style) or Prompt / Response style
        user_col = next((c for c in ["instruction", "prompt", "question", "user", "input"] if c in cols), cols[0])
        asst_col = next((c for c in ["output", "response", "completion", "answer", "assistant"] if c in cols), cols[-1])
        sys_col  = next((c for c in ["system", "system_prompt"] if c in cols), None)
        extra_input_col = "input" if ("instruction" in cols and "input" in cols) else None

        texts = []
        for i in range(len(examples[user_col])):
            convo = []
            if sys_col and examples[sys_col][i]:
                convo.append({"role": "system", "content": str(examples[sys_col][i])})
            
            user_text = str(examples[user_col][i])
            if extra_input_col and examples[extra_input_col][i]:
                user_text += f"\n\n{examples[extra_input_col][i]}"
            
            convo.append({"role": "user", "content": user_text})
            convo.append({"role": "assistant", "content": str(examples[asst_col][i])})
            texts.append(tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False))
        
        return {"text": texts}

    dataset = dataset.map(format_prompts, batched=True)

    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset,
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        dataset_num_proc = 1, # Keep at 1 on Windows to avoid multiprocessing freeze
        packing = False,
        args = TrainingArguments(
            per_device_train_batch_size = 4,
            gradient_accumulation_steps = 4,
            warmup_steps = 10,
            num_train_epochs = num_train_epochs,
            max_steps = max_steps,
            learning_rate = 2e-4,
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 10,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = f"checkpoints_{output_adapter_dir}",
            report_to = "none",
        ),
    )

    trainer_stats = trainer.train()
    
    model.save_pretrained(output_adapter_dir)
    tokenizer.save_pretrained(output_adapter_dir)
    print(f"\n[SUCCESS] Saved LoRA adapter to ./{output_adapter_dir}")

    del trainer, model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return trainer_stats

## Step 4: Train Adapter 1 — `lora_generation_adapter`
Trains the question generation adapter (`BKT mastery + RAG Context -> Question + CO-PO Mapping + Rubric`).

In [10]:
gen_stats = train_ergasia_adapter(
    jsonl_path = "generation_data.jsonl",
    output_adapter_dir = "lora_generation_adapter",
    num_train_epochs = 3
)


Starting training for: lora_generation_adapter
==((====))==  Unsloth 2026.10.5: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.988 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 339/339 [00:03<00:00, 102.29it/s]
Unsloth: Restored added_tokens_decoder metadata in c:\Users\student\Desktop\ergasia_code\ergasia\finetuning\_unsloth_sentencepiece_temp\tokenizer_eydwi_2h\tokenizer_config.json.


Detected columns in generation_data.jsonl: ['text']


Map: 100%|██████████| 18/18 [00:00<00:00, 873.14 examples/s]


Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"]: 100%|██████████| 18/18 [00:00<00:00, 903.67 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18 | Num Epochs = 3 | Total steps = 6
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss


Unsloth: Restored added_tokens_decoder metadata in checkpoints_lora_generation_adapter\checkpoint-6\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in lora_generation_adapter\tokenizer_config.json.



[SUCCESS] Saved LoRA adapter to ./lora_generation_adapter


## Step 5: Train Adapter 2 — `lora_grading_adapter`
Automatically clears VRAM, reloads a clean base model, and trains the automated grading adapter (`Question + Rubric + Student Answer -> Score + Feedback`).

In [11]:
grade_stats = train_ergasia_adapter(
    jsonl_path = "grading_data.jsonl",
    output_adapter_dir = "lora_grading_adapter",
    num_train_epochs = 3
)


Starting training for: lora_grading_adapter
==((====))==  Unsloth 2026.10.5: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.988 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 339/339 [00:03<00:00, 98.58it/s] 
Unsloth: Restored added_tokens_decoder metadata in c:\Users\student\Desktop\ergasia_code\ergasia\finetuning\_unsloth_sentencepiece_temp\tokenizer_vjnhcdny\tokenizer_config.json.
Generating train split: 18 examples [00:00, 2536.71 examples/s]


Detected columns in grading_data.jsonl: ['text']


Map: 100%|██████████| 18/18 [00:00<00:00, 1633.40 examples/s]


Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"]: 100%|██████████| 18/18 [00:00<00:00, 818.20 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18 | Num Epochs = 3 | Total steps = 6
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss


Unsloth: Restored added_tokens_decoder metadata in checkpoints_lora_grading_adapter\checkpoint-6\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in lora_grading_adapter\tokenizer_config.json.



[SUCCESS] Saved LoRA adapter to ./lora_grading_adapter


## Step 6: Smoke Test — Hot-Swapping or Loading Trained Adapters for Inference
Quick verification cell to test loading your newly trained `lora_generation_adapter` for fast local inference.

In [12]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_generation_adapter",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "You are Ergasia AI, an engineering education assistant that generates CO-PO mapped questions and grading rubrics in strict JSON format."},
    {"role": "user", "content": "Generate a question for CO1 at the 'Apply - Analyze' Bloom's level (BKT mastery: 0.55) based on Process Synchronization and Semaphores."}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.2)
print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))

==((====))==  Unsloth 2026.10.5: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.988 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 339/339 [00:03<00:00, 100.95it/s]
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "question": {
    "text": "Given a system with two processes, P1 and P2, that share a critical section of code. The critical section is protected by a binary semaphore `sem` initialized to 1. Describe how you would modify the semaphore operations to ensure mutual exclusion when both processes attempt to enter the critical section simultaneously. Provide the pseudocode for the modified entry and exit sections of the processes.",
    "co_po_mapping": {
      "CO1": "Apply knowledge of synchronization mechanisms",
      "PO1": "Understand and apply process synchronization techniques"
    },
    "bloom_level": "Analyze",
    "mastery_level": 0.55,
    "grading_rubric": [
      {
        "criteria": "Correctly identify the need for a binary semaphore to manage access to the critical section.",
        "points": 2,
        "description": "The student correctly identifies the use of a binary semaphore."
      },
      {
        "criteria": "Provide correct pseudocode for the entry